# TM customer-month tables — exploratory review

**PNC Treasury Management · DSI · Payment Knowledge Graph (PKG)**

Which columns of `pkg_tm_cust_month` (and the related deposit, product and customer-fact tables) are
usable, and for what? This notebook is **read-only**: nothing is written to any database. Tables stay
inline (one output per cell); figures are saved to `OUT_DIR`, never shown inline.

| § | Question |
|---|---|
| 0 | Schemas, months, row counts |
| 1 | Grain and coverage: one row per customer-month? which PKG TM organisations are in it? flags |
| 2 | Column profile: nulls, zeros, negatives, distinct values, range; null share by month |
| 3 | How often each column changes month to month; are the `_1m / _3m / _12m` columns lagged copies? |
| 4 | Identities and meaning: revenue parts, fees (analysed / waived / ECA / billed), FTP and ECR rates, PD grade vs %, exposures |
| 5 | Deposits table: same profile, balance identities, and overlap with the TM balances |
| 6 | Product table: grain and products |
| 7 | Customer fact view: month keys, monthly revenue columns, PD refresh cadence, link to `pkg_tm_cust_month` |
| 8 | Verdict per column: use / lagged copy / duplicate / empty / constant / mostly zero / slow / 12-month smoothed |

In [ ]:
# =============================================================================
# SETUP — environment shims, CONFIG, Spark session, helpers
# =============================================================================
import os, re, sys, time, math, warnings, threading, contextlib
import numpy as np
import pandas as pd

for _a, _t in [("object0", object), ("bool8", np.bool_), ("int0", np.intp),
               ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _a):
        setattr(np, _a, _t)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _w in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_w)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import NumericType, StringType, BooleanType

DB = "bdahd01p_dlcdi1_cdi_tm"
CONFIG = dict(
    TM_TABLE   = f"{DB}.pkg_tm_cust_month",
    DEP_TABLE  = f"{DB}.pkg_dep_cust_month",
    PROD_TABLE = f"{DB}.pkg_tm_cust_prod_month",
    MET_TABLE  = f"{DB}.pkg_cust_metrics_monthly_v3",     # PKG TM organisations, for coverage
    FACT_TABLE = "clhhd01p_clh.cdm_customer_fact_v",
    ID = "customer_pwr_id", MONTH = "month",
    RUN_FACT   = True,        # customer fact view (§7): a scan of a large view, columns pruned
    FACT_LAST_KEYS = 14,      # month keys of the fact view used for the monthly-vs-12-month checks
    CORR_SAMPLE_ROWS = 400_000,
    OUT_DIR = "../eda/tm_cust_month",
    SHUFFLE_PARTITIONS = 400,
)
ID, MONTH = CONFIG["ID"], CONFIG["MONTH"]

spark = (SparkSession.builder.appName("pkg_tm_cust_month_eda")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g")
         .config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g")
         .config("spark.executor.memoryOverhead", "1g")
         .config("spark.yarn.am.memory", "4g")
         .config("spark.yarn.am.memoryOverhead", "1g")
         .config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.kryoserializer.buffer.max", "128m")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
               "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
               "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    pass
spark.conf.set("spark.sql.shuffle.partitions", str(CONFIG["SHUFFLE_PARTITIONS"]))

try:
    from IPython.display import display, HTML
    _NB = "ipykernel" in sys.modules
except Exception:
    _NB = False

def _fmt(v):
    if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "yes" if v else "no"
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        a = abs(float(v))
        if a >= 1000:
            return f"{v:,.0f}"
        if a >= 10:
            return f"{v:,.1f}"
        return f"{v:,.3f}".rstrip("0").rstrip(".") if a else "0"
    return str(v)


def _html(df):
    d = df.copy()
    for c in d.columns:
        d[c] = d[c].map(_fmt)
    return d.to_html(index=False, border=0, escape=True)


class _Cell:
    """Progress lines and result tables of one cell, rendered as a single output."""
    def __init__(self):
        self.lines, self.tables, self.h = [], [], None

    def start(self, title):
        self.title, self.lines, self.tables = title, [], []
        self.h = display(HTML(""), display_id=True) if _NB else None
        self._draw()

    def log(self, txt, replace_last=False):
        if replace_last and self.lines:
            self.lines[-1] = txt
        else:
            self.lines.append(txt)
        self._draw()
        if not _NB:
            print("   ", txt, flush=True)

    def show(self, *blocks):
        self.tables = list(blocks)
        self._draw()
        if not _NB:
            for lab, df in blocks:
                print(f"\n== {lab}")
                print(df.to_string(index=False, max_rows=80))

    def _draw(self):
        if self.h is None:
            return
        css = ("<style>.pkgc table{border-collapse:collapse;font-size:12px;margin:2px 0 12px}"
               ".pkgc th{background:#f1f0ec;color:#0b0b0b;text-align:left;padding:3px 8px;border-bottom:1px solid #b8b7b0}"
               ".pkgc td{padding:2px 8px;border-bottom:1px solid #e6e5e0}"
               ".pkgc h4{margin:10px 0 3px;font-size:13px}.pkgc .pg{color:#8a8985;font-size:12px}</style>")
        body = f"<div class='pkgc'><h3 style='margin:4px 0'>{self.title}</h3>"
        body += "".join(f"<div class='pg'>{x}</div>" for x in self.lines)
        body += "".join(f"<h4>{lab}</h4>{_html(df)}" for lab, df in self.tables)
        self.h.update(HTML(css + body + "</div>"))


CELL = _Cell()
TIMINGS = []


@contextlib.contextmanager
def step(name):
    """Timed step with a live line: elapsed minutes and Spark tasks of the running stage."""
    t0 = time.time()
    CELL.log(f"running: {name} (started {time.strftime('%H:%M')})")
    stop = threading.Event()

    def _tick():
        while not stop.wait(30):
            msg = f"running: {name} ({(time.time() - t0) / 60:.1f} min"
            try:
                st = spark.sparkContext.statusTracker()
                ids = st.getActiveStageIds()
                if ids:
                    info = st.getStageInfo(max(ids))
                    msg += f", stage {info.stageId}: {info.numCompletedTasks:,}/{info.numTasks:,} tasks"
            except Exception:
                pass
            if _NB:
                CELL.log(msg + ")", replace_last=True)

    th = threading.Thread(target=_tick, daemon=True)
    th.start()
    try:
        yield
    finally:
        stop.set()
        mins = (time.time() - t0) / 60
        TIMINGS.append((name, round(mins, 2)))
        if _NB:
            CELL.log(f"done: {name} ({mins:.1f} min)", replace_last=True)
        else:
            print(f"    done: {name} ({mins:.1f} min)", flush=True)


GATES = []


def gate(name, ok, detail, action=""):
    GATES.append(dict(gate=name, result="PASS" if ok else "FAIL", detail=detail, action=action))


def gates_frame():
    return pd.DataFrame(GATES, columns=["gate", "result", "detail", "action"])



OUT_DIR = CONFIG["OUT_DIR"]
os.makedirs(OUT_DIR, exist_ok=True)
FIG_INDEX = []
INK, INK2, GRID, SURF = "#0b0b0b", "#52514e", "#e6e5e0", "#fcfcfb"
C1, C2 = "#2a78d6", "#eb6834"
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": "#b8b7b0", "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.8, "axes.axisbelow": True, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110,
})


def savefig(fig, name, caption):
    fig.savefig(os.path.join(OUT_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    FIG_INDEX.append(dict(file=f"{name}.png", caption=caption))


def mi_col(c):
    c = F.col(c) if isinstance(c, str) else c
    return (F.substring(c, 1, 4).cast("int") * 12 + F.substring(c, 6, 2).cast("int")).cast("int")


def table_months(tbl, col="month"):
    try:
        p = spark.sql(f"SHOW PARTITIONS {tbl}").toPandas().iloc[:, 0]
        m = sorted({x.split(f"{col}=")[1].split("/")[0] for x in p if f"{col}=" in x})
        if m:
            return m
    except Exception:
        pass
    return sorted(str(r[0]) for r in spark.table(tbl).select(col).distinct().collect())


def kinds(sdf):
    """numeric / string / other columns of a Spark frame (booleans counted as numeric)."""
    num, txt, oth = [], [], []
    for f in sdf.schema.fields:
        if isinstance(f.dataType, (NumericType, BooleanType)):
            num.append(f.name)
        elif isinstance(f.dataType, StringType):
            txt.append(f.name)
        else:
            oth.append(f.name)
    return num, txt, oth


def isnull(sdf, c):
    """null, or NaN for float columns (pandas-loaded tables can carry NaN instead of null)."""
    t = dict(sdf.dtypes)[c]
    return (F.col(c).isNull() | F.isnan(c)) if t in ("double", "float") else F.col(c).isNull()


def has(cols, *names):
    return all(n in cols for n in names)


def within(a, b, rel=0.01, abs_=1.0):
    """a and b agree within max(abs_, rel x |b|); both present."""
    return F.abs(F.col(a) - b) <= F.greatest(F.lit(abs_), F.lit(rel) * F.abs(b))


VERDICT_NOTES = {}      # (table, column) -> list of findings collected along the way


def note(tbl, col, txt):
    VERDICT_NOTES.setdefault((tbl, col), []).append(txt)

## §0 · Schemas, months, row counts

In [ ]:
CELL.start("§0 · Inputs")
TABLES = {"tm": CONFIG["TM_TABLE"], "dep": CONFIG["DEP_TABLE"], "prod": CONFIG["PROD_TABLE"],
          "metrics": CONFIG["MET_TABLE"], "fact": CONFIG["FACT_TABLE"]}
SCHEMA, _rows = {}, []
with step("schemas and months"):
    for k, t in TABLES.items():
        try:
            sdf = spark.table(t)
        except Exception as ex:
            _rows.append(dict(table=k, name=t, status=f"not readable: {type(ex).__name__}"))
            continue
        SCHEMA[k] = sdf
        num, txt, oth = kinds(sdf)
        mt = ""
        if MONTH in sdf.columns and k != "fact":
            ms = table_months(t)
            mt = f"{ms[0]} .. {ms[-1]} ({len(ms)})" if ms else ""
        _rows.append(dict(table=k, name=t, status="ok", columns=len(sdf.columns), numeric=len(num),
                          text=len(txt), other=len(oth), months=mt))
    _counts = []
    for k in ("tm", "dep", "prod"):
        if k in SCHEMA and MONTH in SCHEMA[k].columns:
            c = SCHEMA[k].groupBy(MONTH).agg(F.count(F.lit(1)).alias("rows")).toPandas()
            c["table"] = k
            _counts.append(c)
    _counts = (pd.concat(_counts).pivot(index=MONTH, columns="table", values="rows").fillna(0).astype(int)
               .reset_index().sort_values(MONTH)) if _counts else pd.DataFrame()
if "tm" not in SCHEMA:
    raise ValueError(f"{CONFIG['TM_TABLE']} is not readable; nothing else to do.")
_sch = pd.DataFrame([dict(table=k, column=f.name, type=f.dataType.simpleString())
                     for k in ("tm", "dep", "prod") if k in SCHEMA for f in SCHEMA[k].schema.fields])
_sch.to_csv(os.path.join(OUT_DIR, "schemas.csv"), index=False)
CELL.show(("Tables", pd.DataFrame(_rows)), ("Rows per month", _counts),
          ("Columns of tm / dep / prod (also in schemas.csv)",
           _sch.groupby("table")["column"].apply(lambda s: ", ".join(s)).reset_index()))

## §1 · Grain and coverage (`pkg_tm_cust_month`)

One row per customer-month? How many PKG TM organisations (`is_tm`, party type O in the customer metrics)
are in it each month, and how many of its customers are not in PKG? What do the flag columns say?

In [ ]:
CELL.start("§1 · Grain and coverage")
tm = SCHEMA["tm"]
TMC = tm.columns
with step("keys, duplicates, flags"):
    _dup = (tm.groupBy(ID, MONTH).agg(F.count(F.lit(1)).alias("n")).groupBy(MONTH)
            .agg(F.count(F.lit(1)).alias("customers"), F.sum((F.col("n") > 1).cast("long")).alias("duplicated_keys"),
                 F.max("n").alias("max_rows_per_key"))
            .orderBy(MONTH).toPandas())
    _null_id = tm.where(F.col(ID).isNull()).count()
    _flags = [c for c in ["in_ofs", "had_ofs_prev", "cust_gone", "in_cust_fact", "in_pkg_dim", "is_default"] if c in TMC]
    _fl = (tm.groupBy(MONTH).agg(*[F.avg(F.col(c).cast("double")).alias(c) for c in _flags]).orderBy(MONTH).toPandas()
           if _flags else pd.DataFrame())
    _combo = (tm.groupBy(*[c for c in ["in_ofs", "in_cust_fact", "in_pkg_dim", "cust_gone"] if c in TMC])
              .agg(F.count(F.lit(1)).alias("customer_months")).orderBy(F.desc("customer_months")).toPandas())
    _nfr = (tm.groupBy("n_fact_rows").agg(F.count(F.lit(1)).alias("customer_months")).orderBy("n_fact_rows")
            .limit(15).toPandas() if "n_fact_rows" in TMC else pd.DataFrame())
    _cov = pd.DataFrame()
    if "metrics" in SCHEMA and has(SCHEMA["metrics"].columns, "customer_pwr_id", "is_tm", "party_type", "month"):
        _pk = (SCHEMA["metrics"].where((F.col("is_tm").cast("boolean") == F.lit(True)) & (F.col("party_type") == "O"))
               .select(F.col("customer_pwr_id").cast("string").alias(ID), F.col("month").alias(MONTH)).distinct()
               .withColumn("in_pkg_metrics", F.lit(1)))
        _cov = (tm.select(F.col(ID).cast("string").alias(ID), MONTH).distinct().withColumn("in_tm", F.lit(1))
                .join(_pk, [ID, MONTH], "full")
                .groupBy(MONTH).agg(F.sum(F.coalesce("in_tm", F.lit(0))).alias("tm_customers"),
                                    F.sum(F.coalesce("in_pkg_metrics", F.lit(0))).alias("pkg_tm_orgs"),
                                    F.sum((F.col("in_tm").isNotNull() & F.col("in_pkg_metrics").isNotNull()).cast("long")).alias("both"))
                .orderBy(MONTH).toPandas())
        _cov["share_of_pkg_tm_orgs_in_tm"] = _cov["both"] / _cov["pkg_tm_orgs"].where(_cov["pkg_tm_orgs"] > 0)
        _cov["share_of_tm_in_pkg"] = _cov["both"] / _cov["tm_customers"].where(_cov["tm_customers"] > 0)
_g = pd.DataFrame([dict(customer_months=int(_dup.customers.sum()), months=len(_dup),
                        months_with_duplicate_keys=int((_dup.duplicated_keys > 0).sum()),
                        duplicated_keys=int(_dup.duplicated_keys.sum()), null_ids=int(_null_id))])
if _g.duplicated_keys.iloc[0] > 0:
    note("tm", ID, f"{int(_g.duplicated_keys.iloc[0]):,} duplicated customer-months: reduce to one row before joining")
CELL.show(("Grain", _g), ("Coverage against PKG TM organisations (customer metrics)", _cov),
          ("Flag shares by month", _fl), ("Flag combinations (all months)", _combo),
          ("n_fact_rows (fact rows behind a customer-month)", _nfr))

## §2 · Column profile

Every column over all months: share null, share zero and negative (of non-null), approximate distinct
values, and the 1st / 50th / 99th percentile and maximum. Text columns get their most frequent values.
Saved: f01 (share null by month), so a column that starts late or breaks in one month shows up.

In [ ]:
CELL.start("§2 · Column profile")


def profile(sdf, key):
    num, txt, oth = kinds(sdf)
    n = sdf.count()
    aggs = []
    for c in sdf.columns:
        aggs += [F.sum(isnull(sdf, c).cast("long")).alias(f"{c}__null"),
                 F.approx_count_distinct(F.when(~isnull(sdf, c), F.col(c)), 0.05).alias(f"{c}__nd")]
    for c in num:
        x = F.when(~isnull(sdf, c), F.col(c).cast("double"))
        aggs += [F.sum((x == 0).cast("long")).alias(f"{c}__zero"), F.sum((x < 0).cast("long")).alias(f"{c}__neg"),
                 F.percentile_approx(x, [0.01, 0.5, 0.99], 1000).alias(f"{c}__q"),
                 F.max(x).alias(f"{c}__max")]
    r = sdf.agg(*aggs).toPandas().iloc[0]
    rows = []
    for c in sdf.columns:
        nn = n - int(r[f"{c}__null"])
        d = dict(table=key, column=c, kind="number" if c in num else "text" if c in txt else "other",
                 null_share=1 - nn / max(n, 1), distinct=int(r[f"{c}__nd"]))
        if c in num and nn:
            q = r[f"{c}__q"] if r[f"{c}__q"] is not None else [np.nan] * 3
            d.update(zero_share=int(r[f"{c}__zero"]) / nn, neg_share=int(r[f"{c}__neg"]) / nn,
                     p01=q[0], p50=q[1], p99=q[2], max=r[f"{c}__max"])
        rows.append(d)
    prof = pd.DataFrame(rows)
    tops = []
    for c in txt:
        if c in (ID,) or c.endswith("_id") or prof.loc[prof.column == c, "distinct"].iloc[0] > 5000:
            continue
        t = sdf.groupBy(c).agg(F.count(F.lit(1)).alias("n")).orderBy(F.desc("n")).limit(8).toPandas()
        tops.append(dict(table=key, column=c, top_values=", ".join(f"{v} ({k / n:.1%})" for v, k in zip(t[c], t["n"]))))
    return prof, pd.DataFrame(tops), n


def null_by_month(sdf):
    return (sdf.groupBy(MONTH).agg(*[F.avg(isnull(sdf, c).cast("double")).alias(c) for c in sdf.columns if c != MONTH])
            .orderBy(MONTH).toPandas().set_index(MONTH))


def heat(nbm, name, title):
    m = nbm.T
    fig, ax = plt.subplots(figsize=(min(18, 4 + 0.32 * m.shape[1]), 0.24 * m.shape[0] + 1.6))
    im = ax.imshow(m.to_numpy(float), aspect="auto", cmap="Blues", vmin=0, vmax=1, interpolation="nearest")
    ax.set_yticks(range(m.shape[0])); ax.set_yticklabels(m.index, fontsize=8)
    ax.set_xticks(range(m.shape[1])); ax.set_xticklabels(m.columns, rotation=90, fontsize=7)
    ax.grid(False); ax.set_title(title)
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01); cb.set_label("share null")
    savefig(fig, name, title)


with step("profile pkg_tm_cust_month"):
    PROF_TM, TOPS_TM, N_TM = profile(tm, "tm")
    NBM_TM = null_by_month(tm)
heat(NBM_TM, "f01_tm_null_by_month", "pkg_tm_cust_month: share null by column and month")
PROF_TM.to_csv(os.path.join(OUT_DIR, "profile_tm.csv"), index=False)
CELL.show((f"pkg_tm_cust_month: {N_TM:,} rows", PROF_TM.drop(columns=["table"])),
          ("Most frequent values of text columns", TOPS_TM.drop(columns=["table"]) if len(TOPS_TM) else TOPS_TM))

## §3 · Change cadence and lagged copies

For each customer and each pair of consecutive months: did the value change? A column that changes in a
small share of months is refreshed rarely (PD reviews, for example) and cannot time an event to the month.
Then: is `pd_pct_1m` this month the same as `pd_pct_now` one month earlier (likewise `_3m`, `_12m`, and the
grades)? If so, those columns are lagged copies and can be dropped from any table we keep.
Saved: f02 (share of months with a change, per column).

In [ ]:
CELL.start("§3 · Change cadence and lagged copies")


def change_share(sdf, key):
    num, txt, _ = kinds(sdf)
    cols = [c for c in num + txt if c not in (ID, MONTH)]
    w = W.partitionBy(ID).orderBy("_mi")
    s = sdf.select(F.col(ID).cast("string").alias(ID), mi_col(MONTH).alias("_mi"),
                   *[F.when(~isnull(sdf, c), F.col(c)).alias(c) for c in cols])
    s = s.withColumn("_lmi", F.lag("_mi").over(w))
    for c in cols:
        s = s.withColumn(f"_l_{c}", F.lag(c).over(w))
    cons = F.col("_lmi") == F.col("_mi") - 1
    aggs = []
    for c in cols:
        both = cons & F.col(c).isNotNull() & F.col(f"_l_{c}").isNotNull()
        aggs += [F.sum(both.cast("long")).alias(f"{c}__pairs"),
                 F.sum((both & (F.col(c) != F.col(f"_l_{c}"))).cast("long")).alias(f"{c}__chg")]
    r = s.agg(*aggs).toPandas().iloc[0]
    return pd.DataFrame([dict(table=key, column=c, month_pairs=int(r[f"{c}__pairs"]),
                              change_share=int(r[f"{c}__chg"]) / int(r[f"{c}__pairs"]) if int(r[f"{c}__pairs"]) else np.nan)
                         for c in cols])


def lag_copies(sdf, key, pairs):
    """pairs: (copy column, source column, lag in months). Share where copy(m) == source(m - lag)."""
    pairs = [p for p in pairs if has(sdf.columns, p[0], p[1])]
    if not pairs:
        return pd.DataFrame()
    w = W.partitionBy(ID).orderBy("_mi")
    s = sdf.select(F.col(ID).cast("string").alias(ID), mi_col(MONTH).alias("_mi"),
                   *sorted({p[0] for p in pairs} | {p[1] for p in pairs}))
    aggs = []
    for i, (cp, src, k) in enumerate(pairs):
        s = s.withColumn(f"_lm{i}", F.lag("_mi", k).over(w)).withColumn(f"_lv{i}", F.lag(src, k).over(w))
        ok = (F.col(f"_lm{i}") == F.col("_mi") - k) & F.col(cp).isNotNull() & F.col(f"_lv{i}").isNotNull()
        same = (F.abs(F.col(cp).cast("double") - F.col(f"_lv{i}").cast("double")) <= 1e-9) \
            if dict(sdf.dtypes)[cp] != "string" else (F.col(cp) == F.col(f"_lv{i}"))
        aggs += [F.sum(ok.cast("long")).alias(f"p{i}"), F.sum((ok & same).cast("long")).alias(f"m{i}"),
                 F.sum((F.col(cp).isNotNull() & F.col(src).isNotNull() & (F.col(cp) == F.col(src))).cast("long")).alias(f"e{i}"),
                 F.sum((F.col(cp).isNotNull() & F.col(src).isNotNull()).cast("long")).alias(f"b{i}")]
    r = s.agg(*aggs).toPandas().iloc[0]
    out = []
    for i, (cp, src, k) in enumerate(pairs):
        p, m, e, b = (int(r[x]) for x in (f"p{i}", f"m{i}", f"e{i}", f"b{i}"))
        out.append(dict(table=key, column=cp, compared_with=f"{src}, {k} month(s) earlier", rows_compared=p,
                        match_share=m / p if p else np.nan, same_month_equal_share=e / b if b else np.nan))
    return pd.DataFrame(out)


TM_LAGS = [("pd_pct_1m", "pd_pct_now", 1), ("pd_pct_3m", "pd_pct_now", 3), ("pd_pct_12m", "pd_pct_now", 12),
           ("pd_grade_1m", "pd_grade_now", 1), ("pd_grade_3m", "pd_grade_now", 3), ("pd_grade_12m", "pd_grade_now", 12),
           ("had_ofs_prev", "in_ofs", 1)]
with step("change cadence (pkg_tm_cust_month)"):
    CHG_TM = change_share(tm, "tm")
with step("lagged copies"):
    LAG_TM = lag_copies(tm, "tm", TM_LAGS)
for r in LAG_TM.itertuples():
    if r.match_share >= 0.98:
        note("tm", r.column, f"lagged copy: equals {r.compared_with} in {r.match_share:.1%} of rows")
_c = CHG_TM.sort_values("change_share")
fig, ax = plt.subplots(figsize=(8, 0.26 * len(_c) + 1.2))
ax.barh(_c.column, 100 * _c.change_share, color=C1, height=0.7)
ax.set_xlabel("% of consecutive customer-months with a different value"); ax.grid(axis="y", visible=False)
ax.set_title("pkg_tm_cust_month: how often each column changes")
ax.tick_params(axis="y", labelsize=8)
savefig(fig, "f02_tm_change_share", "pkg_tm_cust_month: share of consecutive months with a change, per column")
CELL.show(("Share of consecutive customer-months with a change (low = refreshed rarely)",
           CHG_TM.drop(columns=["table"]).sort_values("change_share")),
          ("Lagged copies: does copy(m) equal source(m - lag)?", LAG_TM.drop(columns=["table"]) if len(LAG_TM) else LAG_TM))

## §4 · Identities and meaning (`pkg_tm_cust_month`)

Each check runs only if its columns exist.

- **Revenue:** gross = credit + non-credit? TM revenue within non-credit?
- **Fees:** which combination of analysed charges, waived, ECA used and ECR equals billed?
- **Rates:** FTP value x 12 / average balance and ECR amount x 12 / investable balance, against `ecr_rate_wavg`.
- **Balances:** investable / average balance.
- **Credit:** PD % range per grade, PD of defaulted customers, exposures (DHE, WRRE, MRE, soft, watchlist).
- **Duplicates:** rank correlation of every pair of numeric columns on a customer sample, in levels and in
  month-to-month changes. Two columns that only rank customers alike (big customers have big balances and big
  revenue) are not duplicates; a duplicate also moves together month to month.

In [ ]:
CELL.start("§4 · Identities and meaning")
_chk = []


def check(name, cond_expr, base_expr, why):
    r = tm.agg(F.sum(base_expr.cast("long")).alias("b"), F.sum((base_expr & cond_expr).cast("long")).alias("m")).toPandas().iloc[0]
    b, m = int(r.b or 0), int(r.m or 0)
    _chk.append(dict(check=name, rows=b, holds_share=m / b if b else np.nan, meaning=why))
    return m / b if b else np.nan


with step("identities"):
    nn = lambda *cs: F.lit(True) if not cs else (F.col(cs[0]).isNotNull() & nn(*cs[1:]))
    if has(TMC, "gross_rev_12m", "credit_rev_12m", "non_credit_rev_12m"):
        check("gross_rev_12m = credit_rev_12m + non_credit_rev_12m",
              within("gross_rev_12m", F.col("credit_rev_12m") + F.col("non_credit_rev_12m")),
              nn("gross_rev_12m", "credit_rev_12m", "non_credit_rev_12m"), "if it holds, gross is redundant")
    if has(TMC, "tm_rev_12m", "non_credit_rev_12m"):
        check("tm_rev_12m <= non_credit_rev_12m", F.col("tm_rev_12m") <= F.col("non_credit_rev_12m") + 1,
              nn("tm_rev_12m", "non_credit_rev_12m"), "TM revenue is part of non-credit revenue")
    if has(TMC, "billed", "analyzed_chrg"):
        for lab, expr, cs in [("billed = analyzed_chrg", F.col("analyzed_chrg"), ["analyzed_chrg"]),
                              ("billed = analyzed_chrg - waived", F.col("analyzed_chrg") - F.col("waived"), ["waived"]),
                              ("billed = analyzed_chrg - eca_used", F.col("analyzed_chrg") - F.col("eca_used"), ["eca_used"]),
                              ("billed = analyzed_chrg - waived - eca_used",
                               F.col("analyzed_chrg") - F.col("waived") - F.col("eca_used"), ["waived", "eca_used"]),
                              ("billed = max(analyzed_chrg - waived - ecr_amt, 0)",
                               F.greatest(F.col("analyzed_chrg") - F.col("waived") - F.col("ecr_amt"), F.lit(0.0)), ["waived", "ecr_amt"])]:
            if has(TMC, *cs):
                check(lab, within("billed", expr), nn("billed", "analyzed_chrg", *cs) & (F.col("analyzed_chrg") > 0),
                      "the matching formula tells which fee column is revenue actually collected")
    if has(TMC, "eca_used", "ecr_amt"):
        check("eca_used <= ecr_amt", F.col("eca_used") <= F.col("ecr_amt") + 1, nn("eca_used", "ecr_amt") & (F.col("ecr_amt") > 0),
              "earnings credit used cannot exceed the credit earned")
    if has(TMC, "n_acct", "n_acct_open"):
        check("n_acct_open <= n_acct", F.col("n_acct_open") <= F.col("n_acct"), nn("n_acct", "n_acct_open"), "account counts")
    if has(TMC, "inv_bal", "avg_bal"):
        check("inv_bal <= avg_bal", F.col("inv_bal") <= F.col("avg_bal") + 1, nn("inv_bal", "avg_bal") & (F.col("avg_bal") > 0),
              "investable balance = average balance less float and reserves")

    _rates = []
    def _ratio(lab, num_expr, den_col, extra=None):
        base = F.col(den_col) > 1000
        if extra is not None:
            base = base & extra
        q = (tm.where(base).select((num_expr / F.col(den_col)).alias("x"))
             .agg(F.expr("percentile_approx(x, array(0.1, 0.25, 0.5, 0.75, 0.9), 1000)").alias("q"), F.count(F.lit(1)).alias("n"))
             .toPandas().iloc[0])
        if q.q is not None:
            _rates.append(dict(ratio=lab, rows=int(q.n), p10=q.q[0], p25=q.q[1], p50=q.q[2], p75=q.q[3], p90=q.q[4]))
    if has(TMC, "ftp_value", "avg_bal"):
        _ratio("ftp_value x 12 / avg_bal (annual rate if ftp_value is monthly)", F.col("ftp_value") * 12, "avg_bal")
    if has(TMC, "ecr_amt", "inv_bal"):
        _ratio("ecr_amt x 12 / inv_bal (annual earnings-credit rate)", F.col("ecr_amt") * 12, "inv_bal")
    if has(TMC, "ecr_rate_wavg"):
        q = tm.where(F.col("ecr_rate_wavg").isNotNull()).agg(
            F.expr("percentile_approx(ecr_rate_wavg, array(0.1, 0.25, 0.5, 0.75, 0.9), 1000)").alias("q"),
            F.count(F.lit(1)).alias("n")).toPandas().iloc[0]
        if q.q is not None:
            _rates.append(dict(ratio="ecr_rate_wavg (as stored)", rows=int(q.n), p10=q.q[0], p25=q.q[1], p50=q.q[2], p75=q.q[3], p90=q.q[4]))
    if has(TMC, "inv_bal", "avg_bal"):
        _ratio("inv_bal / avg_bal", F.col("inv_bal"), "avg_bal")
    if has(TMC, "analyzed_chrg", "avg_bal"):
        _ratio("analyzed_chrg x 12 / avg_bal (fees as a rate on balances)", F.col("analyzed_chrg") * 12, "avg_bal")
    _rates = pd.DataFrame(_rates)

    _pdg = pd.DataFrame()
    if has(TMC, "pd_grade_now", "pd_pct_now"):
        _pdg = (tm.where(F.col("pd_grade_now").isNotNull()).groupBy("pd_grade_now")
                .agg(F.count(F.lit(1)).alias("customer_months"), F.min("pd_pct_now").alias("pd_min"),
                     F.expr("percentile_approx(pd_pct_now, 0.5, 1000)").alias("pd_median"), F.max("pd_pct_now").alias("pd_max"),
                     F.avg(F.col("is_default").cast("double")).alias("default_share") if "is_default" in TMC else F.lit(None).alias("default_share"))
                .orderBy("pd_median").toPandas())
    _dft = pd.DataFrame()
    if has(TMC, "is_default", "pd_pct_now"):
        _dft = (tm.groupBy(F.col("is_default").alias("is_default"))
                .agg(F.count(F.lit(1)).alias("customer_months"),
                     F.expr("percentile_approx(pd_pct_now, array(0.1, 0.5, 0.9), 1000)").alias("pd_p10_p50_p90"),
                     *([F.avg((F.col("past_due_days") > 0).cast("double")).alias("past_due_share")] if "past_due_days" in TMC else []))
                .toPandas())
        _dft["pd_p10_p50_p90"] = _dft["pd_p10_p50_p90"].map(lambda q: ", ".join(_fmt(v) for v in q) if q is not None else "")
    _exp = [c for c in ["dhe", "wrre", "mre", "soft_exposure", "watchlist_exposure", "past_due_days"] if c in TMC]
    _ex = pd.DataFrame()
    if _exp:
        r = tm.agg(*[F.avg((F.col(c) > 0).cast("double")).alias(c) for c in _exp]).toPandas().iloc[0]
        _ex = pd.DataFrame([dict(column=c, share_above_zero=r[c]) for c in _exp])
        if "credit_exposure_type" in TMC:
            _cet = (tm.groupBy("credit_exposure_type").agg(F.count(F.lit(1)).alias("n"),
                                                           *[F.avg((F.col(c) > 0).cast("double")).alias(f"{c}>0") for c in _exp])
                    .orderBy(F.desc("n")).limit(10).toPandas())
        else:
            _cet = pd.DataFrame()

with step("rank correlations: levels and month-to-month changes (customer sample)"):
    _num = [c for c in kinds(tm)[0] if c not in (ID, MONTH)]
    _pct = int(min(100, math.ceil(100 * CONFIG["CORR_SAMPLE_ROWS"] / max(N_TM, 1))))
    _smp = (tm.where(F.abs(F.hash(F.col(ID).cast("string"))) % 100 < _pct)
            .select(F.col(ID).cast("string").alias(ID), mi_col(MONTH).alias("_mi"),
                    *[F.when(~isnull(tm, c), F.col(c).cast("double")).alias(c) for c in _num]).toPandas())
    _keep = [c for c in _num if _smp[c].std() > 0]
    _rho = _smp[_keep].rank().corr()
    _smp = _smp.sort_values([ID, "_mi"])
    _dd = _smp.groupby(ID)[_keep].diff()[_smp.groupby(ID)["_mi"].diff() == 1]
    _rhod = _dd.rank().corr()
    _dups = []
    for i, a in enumerate(_keep):
        for b in _keep[i + 1:]:
            v = _rho.loc[a, b]
            if np.isfinite(v) and abs(v) >= 0.98:
                _dups.append(dict(column_a=a, column_b=b, rank_corr_levels=v, rank_corr_changes=_rhod.loc[a, b]))
    _dups = (pd.DataFrame(_dups, columns=["column_a", "column_b", "rank_corr_levels", "rank_corr_changes"])
             .sort_values(["rank_corr_changes", "rank_corr_levels"], ascending=False))
    for r in _dups.itertuples():
        if abs(r.rank_corr_levels) >= 0.995 and np.isfinite(r.rank_corr_changes) and abs(r.rank_corr_changes) >= 0.98:
            note("tm", r.column_b, f"near-duplicate of {r.column_a} (rank corr {r.rank_corr_levels:.3f} levels, "
                                   f"{r.rank_corr_changes:.3f} monthly changes)")
_chk = pd.DataFrame(_chk)
for r in _chk.itertuples():
    if r.check.startswith("gross_rev_12m =") and r.holds_share >= 0.98:
        note("tm", "gross_rev_12m", "equals credit + non-credit revenue: redundant")
CELL.show(("Identity checks", _chk), ("Ratios (distribution)", _rates), ("PD % by grade", _pdg),
          ("Defaulted vs not", _dft), ("Exposure columns: share above zero", _ex),
          ("Exposure columns by credit_exposure_type", _cet if _exp else pd.DataFrame()),
          (f"Numeric pairs with rank correlation 0.98 or more in levels ({_smp[ID].nunique():,} sampled customers); "
           "a duplicate also moves together month to month", _dups.head(60)))

## §5 · Deposits table (`pkg_dep_cust_month`) and overlap with TM balances

Same profile and change cadence; whether the total balance equals the sum of the product balances; and,
for customer-months in both tables, how TM `avg_bal` / `inv_bal` compare with deposit `dep_bal_mean` and
`bal_dda_mean`. If they match, one of them is redundant. Saved: f03 (null share by month), f04 (TM average
balance against deposit average balance).

In [ ]:
if "dep" in SCHEMA:
    CELL.start("§5 · Deposits table and overlap with TM balances")
    dep = SCHEMA["dep"]
    DPC = dep.columns
    with step("profile and cadence (pkg_dep_cust_month)"):
        PROF_DEP, TOPS_DEP, N_DEP = profile(dep, "dep")
        CHG_DEP = change_share(dep, "dep")
        NBM_DEP = null_by_month(dep)
    heat(NBM_DEP, "f03_dep_null_by_month", "pkg_dep_cust_month: share null by column and month")
    PROF_DEP.to_csv(os.path.join(OUT_DIR, "profile_dep.csv"), index=False)
    _dchk = []
    with step("balance identities and overlap"):
        for kind_ in ("mean", "eom"):
            parts = [c for c in DPC if c.startswith("bal_") and c.endswith(f"_{kind_}")]
            tot = f"dep_bal_{kind_}"
            if parts and tot in DPC:
                s = sum(F.coalesce(F.col(p), F.lit(0.0)) for p in parts)
                r = dep.where(F.col(tot).isNotNull()).agg(F.count(F.lit(1)).alias("b"),
                                                          F.sum(within(tot, s).cast("long")).alias("m")).toPandas().iloc[0]
                _dchk.append(dict(check=f"{tot} = sum of {len(parts)} product balances ({', '.join(parts)})",
                                  rows=int(r.b), holds_share=int(r.m) / max(int(r.b), 1)))
        if has(DPC, "n_days_present", "n_days"):
            r = dep.agg(F.avg((F.col("n_days_present") <= F.col("n_days")).cast("double")).alias("x")).toPandas().iloc[0]
            _dchk.append(dict(check="n_days_present <= n_days", rows=N_DEP, holds_share=float(r.x)))
        _ov, _ovr = pd.DataFrame(), pd.DataFrame()
        tb = [c for c in ["avg_bal", "inv_bal", "n_acct"] if c in TMC]
        db = [c for c in ["dep_bal_mean", "dep_bal_eom", "bal_dda_mean", "n_acct"] if c in DPC]
        if tb and db:
            j = (tm.select(F.col(ID).cast("string").alias(ID), MONTH, *[F.col(c).alias(f"tm_{c}") for c in tb])
                 .join(dep.select(F.col(ID).cast("string").alias(ID), MONTH, *[F.col(c).alias(f"dep_{c}") for c in db]),
                       [ID, MONTH], "full"))
            _ov = (j.groupBy(MONTH).agg(
                       F.sum(F.col(f"tm_{tb[0]}").isNotNull().cast("long")).alias("in_tm"),
                       F.sum(F.col(f"dep_{db[0]}").isNotNull().cast("long")).alias("in_dep"),
                       F.sum((F.col(f"tm_{tb[0]}").isNotNull() & F.col(f"dep_{db[0]}").isNotNull()).cast("long")).alias("in_both"))
                   .orderBy(MONTH).toPandas())
            both = j.where(F.col(f"tm_{tb[0]}").isNotNull() & F.col(f"dep_{db[0]}").isNotNull())
            rows = []
            for a in [c for c in tb if c != "n_acct"]:
                for b in [c for c in db if c != "n_acct"]:
                    r = (both.where(F.col(f"dep_{b}") > 1000)
                         .agg(F.count(F.lit(1)).alias("n"),
                              F.expr(f"percentile_approx(tm_{a} / dep_{b}, array(0.1, 0.5, 0.9), 1000)").alias("q"),
                              F.avg(within(f"tm_{a}", F.col(f"dep_{b}"), rel=0.05).cast("double")).alias("w5"),
                              F.corr(F.log1p(F.greatest(F.col(f"tm_{a}"), F.lit(0.0))),
                                     F.log1p(F.greatest(F.col(f"dep_{b}"), F.lit(0.0)))).alias("corr_log"))
                         .toPandas().iloc[0])
                    if r.q is not None:
                        rows.append(dict(tm_column=a, dep_column=b, rows=int(r.n), ratio_p10=r.q[0], ratio_p50=r.q[1],
                                         ratio_p90=r.q[2], within_5pct=r.w5, corr_log=r.corr_log))
            if has(tb, "n_acct") and has(db, "n_acct"):
                r = both.agg(F.avg((F.col("tm_n_acct") == F.col("dep_n_acct")).cast("double")).alias("x")).toPandas().iloc[0]
                rows.append(dict(tm_column="n_acct", dep_column="n_acct", rows=None, within_5pct=float(r.x)))
            _ovr = pd.DataFrame(rows)
            for r in _ovr.itertuples():
                if r.tm_column != "n_acct" and np.isfinite(r.within_5pct) and r.within_5pct >= 0.9:
                    note("tm", r.tm_column, f"matches deposits {r.dep_column} within 5% in {r.within_5pct:.0%} of shared rows")
            if "avg_bal" in tb and "dep_bal_mean" in db:
                _s = (both.where((F.col("tm_avg_bal") > 0) & (F.col("dep_dep_bal_mean") > 0))
                      .sample(False, 0.05, seed=3).limit(200_000).select("tm_avg_bal", "dep_dep_bal_mean").toPandas())
                if len(_s):
                    fig, ax = plt.subplots(figsize=(6.2, 5.6))
                    hb = ax.hexbin(np.log10(_s.dep_dep_bal_mean), np.log10(_s.tm_avg_bal), gridsize=60, cmap="Blues", mincnt=1, bins="log")
                    lo, hi = np.percentile(np.log10(_s.dep_dep_bal_mean), [0.5, 99.5])
                    ax.plot([lo, hi], [lo, hi], color=INK2, linewidth=1, linestyle=":")
                    ax.set_xlabel("deposits dep_bal_mean (log10 USD)"); ax.set_ylabel("TM avg_bal (log10 USD)")
                    ax.set_title("Same customer-month: TM average balance vs deposit average balance"); ax.grid(False)
                    fig.colorbar(hb, ax=ax, fraction=0.04, pad=0.01).set_label("customer-months")
                    savefig(fig, "f04_tm_vs_dep_balance", "TM avg_bal against deposits dep_bal_mean, same customer-month (dotted = equal)")
    CELL.show((f"pkg_dep_cust_month: {N_DEP:,} rows", PROF_DEP.drop(columns=["table"])),
              ("Change cadence", CHG_DEP.drop(columns=["table"]).sort_values("change_share")),
              ("Balance identities", pd.DataFrame(_dchk)),
              ("Customer-months in TM, deposits, both", _ov),
              ("TM balances against deposit balances (shared customer-months)", _ovr))

## §6 · Product table (`pkg_tm_cust_prod_month`)

Grain (which columns make a row unique), products, and the generic profile.

In [ ]:
if "prod" in SCHEMA:
    CELL.start("§6 · Product table")
    prod = SCHEMA["prod"]
    PC = prod.columns
    _pcol = next((c for c in ["prod_desc_ofs", "product", "prod_desc", "product_desc"] if c in PC), None)
    if _pcol is None:
        _pcol = next((c for c in PC if "prod" in c.lower() and dict(prod.dtypes)[c] == "string"), None)
    with step("product table"):
        PROF_PROD, TOPS_PROD, N_PROD = profile(prod, "prod")
        _gr = pd.DataFrame()
        if _pcol:
            _gr = (prod.groupBy(ID, MONTH, _pcol).agg(F.count(F.lit(1)).alias("n"))
                   .agg(F.count(F.lit(1)).alias("keys"), F.sum((F.col("n") > 1).cast("long")).alias("duplicated_keys"),
                        F.max("n").alias("max_rows_per_key")).toPandas())
            _gr.insert(0, "key", f"{ID} x {MONTH} x {_pcol}")
            _last = table_months(CONFIG["PROD_TABLE"])[-1]
            _top = (prod.where(F.col(MONTH) == _last).groupBy(_pcol)
                    .agg(F.countDistinct(ID).alias("customers"), F.count(F.lit(1)).alias("rows"))
                    .orderBy(F.desc("customers")).limit(25).toPandas())
            _npm = (prod.groupBy(MONTH).agg(F.countDistinct(_pcol).alias("products"), F.countDistinct(ID).alias("customers"))
                    .orderBy(MONTH).toPandas())
        else:
            _top, _npm = pd.DataFrame(), pd.DataFrame()
    PROF_PROD.to_csv(os.path.join(OUT_DIR, "profile_prod.csv"), index=False)
    CELL.show(("Grain", _gr), (f"Top products in the last month (by customers)", _top), ("Products and customers by month", _npm),
              (f"Profile ({N_PROD:,} rows)", PROF_PROD.drop(columns=["table"])))

## §7 · Customer fact view (`cdm_customer_fact_v`)

Column names are discovered, not assumed: month keys (`time_key`, `as_of_dt`), monthly `_1_mnth` columns with
a `_12_mnth` partner, PD columns. Checks:

- what a month key looks like (values, rows and customers per key) and how it maps to a calendar month;
- whether 12 consecutive monthly values add up to the 12-month value (then the monthly columns are true months);
- how often the 1-month PD changes, and the age of the PD date if there is one (refresh cadence);
- whether the fact values equal `pkg_tm_cust_month` for the same customer-month (lineage).

Saved: f05 (rows per month key), f06 (age of PD in months), if available.

In [ ]:
if CONFIG["RUN_FACT"] and "fact" in SCHEMA:
    CELL.start("§7 · Customer fact view")
    fact = SCHEMA["fact"]
    FC = fact.columns
    fid = next((c for c in ["cust_pwr_id", "customer_pwr_id"] if c in FC), None)
    keys = [c for c in ["time_key", "as_of_dt"] if c in FC]
    pairs = [c[:-len("_1_mnth")] for c in FC if c.endswith("_1_mnth") and c[:-len("_1_mnth")] + "_12_mnth" in FC]
    pdc = [c for c in FC if re.search(r"(^|_)pd(_|$)", c)]
    upg = [c for c in FC if "upg" in c or "dng" in c]
    _disc = pd.DataFrame([dict(item="customer id", columns=fid or "not found"),
                          dict(item="month keys", columns=", ".join(keys) or "not found"),
                          dict(item="monthly columns with a 12-month partner", columns=", ".join(f"{p}_1_mnth" for p in pairs)),
                          dict(item="PD columns", columns=", ".join(pdc)),
                          dict(item="upgrade / downgrade columns", columns=", ".join(upg)),
                          dict(item="all _1_mnth columns", columns=", ".join(c for c in FC if c.endswith("_1_mnth")))])
    if not (fid and keys):
        CELL.show(("Discovery", _disc))
    else:
        K = keys[0]
        with step("month keys"):
            _kc = (fact.groupBy(K).agg(F.count(F.lit(1)).alias("rows"), F.approx_count_distinct(fid, 0.02).alias("customers"),
                                       *[F.min(k).alias(f"min_{k}") for k in keys if k != K],
                                       *[F.max(k).alias(f"max_{k}") for k in keys if k != K])
                   .orderBy(F.desc(K)).toPandas())
            last = sorted(_kc[K].tolist())[-CONFIG["FACT_LAST_KEYS"]:]
            _dpk = (fact.where(F.col(K).isin(last)).groupBy(fid, K).agg(F.count(F.lit(1)).alias("n"))
                    .agg(F.sum((F.col("n") > 1).cast("long")).alias("duplicated_keys"), F.max("n").alias("max_rows")).toPandas())

        def _to_month(colname):
            """yyyymm or yyyymmdd numbers (or dates) -> 'YYYY-MM'; None when the format is something else."""
            s = F.col(colname).cast("string")
            return (F.when(s.rlike(r"^\d{8}$"), F.concat(F.substring(s, 1, 4), F.lit("-"), F.substring(s, 5, 2)))
                    .when(s.rlike(r"^\d{6}$"), F.concat(F.substring(s, 1, 4), F.lit("-"), F.substring(s, 5, 2)))
                    .when(s.rlike(r"^\d{4}-\d{2}"), F.substring(s, 1, 7)))
        _mcol = next((k for k in reversed(keys)), None)
        f12 = (fact.where(F.col(K).isin(last))
               .select(F.col(fid).cast("string").alias(ID), F.col(K).alias("_k"), _to_month(_mcol).alias(MONTH),
                       *[F.col(c).cast("double").alias(c) for c in
                         sorted({f"{p}_1_mnth" for p in pairs} | {f"{p}_12_mnth" for p in pairs}
                                | {f"{p}_3_mnth" for p in pairs if f"{p}_3_mnth" in FC}
                                | set(c for c in pdc if dict(fact.dtypes)[c] not in ("string", "timestamp", "date"))
                                | ({"dhe"} if "dhe" in FC else set()))],
                       *[F.col(c) for c in pdc if dict(fact.dtypes)[c] in ("string", "timestamp", "date")],
                       *[F.col(c) for c in upg if c not in pdc])
               .dropDuplicates([ID, "_k"]).persist())
        with step("monthly vs 12-month sums, PD cadence"):
            _kmap = f12.groupBy("_k").agg(F.first(MONTH, ignorenulls=True).alias(MONTH)).orderBy("_k").toPandas()
            rank = {k: i for i, k in enumerate(sorted(last))}
            f12r = f12.withColumn("_r", F.create_map(*sum([[F.lit(k), F.lit(v)] for k, v in rank.items()], []))[F.col("_k")])
            _sums = []
            w12 = W.partitionBy(ID).orderBy("_r").rangeBetween(-11, 0)
            w3 = W.partitionBy(ID).orderBy("_r").rangeBetween(-2, 0)
            top = max(rank.values())
            for p in pairs[:10]:
                x = (f12r.withColumn("_s12", F.sum(f"{p}_1_mnth").over(w12)).withColumn("_n12", F.count(f"{p}_1_mnth").over(w12))
                     .withColumn("_s3", F.sum(f"{p}_1_mnth").over(w3)).withColumn("_n3", F.count(f"{p}_1_mnth").over(w3))
                     .where(F.col("_r") == top))
                aggs = [F.sum(((F.col("_n12") == 12) & F.col(f"{p}_12_mnth").isNotNull()).cast("long")).alias("b12"),
                        F.sum(((F.col("_n12") == 12) & within(f"{p}_12_mnth", F.col("_s12"), rel=0.02)).cast("long")).alias("m12")]
                if f"{p}_3_mnth" in FC:
                    aggs += [F.sum(((F.col("_n3") == 3) & F.col(f"{p}_3_mnth").isNotNull()).cast("long")).alias("b3"),
                             F.sum(((F.col("_n3") == 3) & within(f"{p}_3_mnth", F.col("_s3"), rel=0.02)).cast("long")).alias("m3")]
                r = x.agg(*aggs).toPandas().iloc[0]
                b3 = int(r["b3"] or 0) if "b3" in r.index else 0
                _sums.append(dict(column=f"{p}_1_mnth", customers_with_12_keys=int(r.b12 or 0),
                                  sum_of_12_equals_12_mnth=(int(r.m12 or 0) / int(r.b12)) if r.b12 else np.nan,
                                  sum_of_3_equals_3_mnth=(int(r["m3"] or 0) / b3) if b3 else np.nan))
            _sums = pd.DataFrame(_sums)
            _pdcad = []
            wk = W.partitionBy(ID).orderBy("_r")
            for c in [c for c in pdc if dict(fact.dtypes)[c] not in ("timestamp", "date")][:6]:
                y = f12r.withColumn("_l", F.lag(c).over(wk)).withColumn("_lr", F.lag("_r").over(wk))
                r = y.where((F.col("_lr") == F.col("_r") - 1) & F.col(c).isNotNull() & F.col("_l").isNotNull()).agg(
                    F.count(F.lit(1)).alias("n"), F.avg((F.col(c) != F.col("_l")).cast("double")).alias("x")).toPandas().iloc[0]
                _pdcad.append(dict(column=c, month_pairs=int(r.n), change_share=r.x))
            _pdcad = pd.DataFrame(_pdcad)
            _age = pd.DataFrame()
            _dtc = next((c for c in pdc if dict(fact.dtypes)[c] in ("timestamp", "date")), None)
            if _dtc:
                ag = (f12r.where(F.col("_r") == top).where(F.col(_dtc).isNotNull() & F.col(MONTH).isNotNull())
                      .select(F.months_between(F.last_day(F.to_date(F.concat(F.col(MONTH), F.lit("-01")))),
                                               F.to_date(F.col(_dtc))).alias("age")).toPandas())
                if len(ag):
                    _age = pd.DataFrame([dict(column=_dtc, customers=len(ag), p25=ag.age.quantile(.25), median=ag.age.median(),
                                              p75=ag.age.quantile(.75), within_1_month=(ag.age <= 1).mean(),
                                              within_3_months=(ag.age <= 3).mean(), within_12_months=(ag.age <= 12).mean())])
                    fig, ax = plt.subplots(figsize=(8, 3.8))
                    ax.hist(ag.age.clip(0, 36), bins=36, color=C1, edgecolor=SURF)
                    ax.set_xlabel("months since PD date (capped at 36)"); ax.set_ylabel("customers")
                    ax.set_title(f"Age of the PD at the latest month key ({_dtc})")
                    savefig(fig, "f06_pd_age", "Months between the latest month key and the PD date")
            _ud = pd.DataFrame()
            if upg:
                c0 = next((c for c in upg if dict(fact.dtypes)[c] == "string"), None)
                if c0:
                    _ud = (f12r.groupBy(MONTH, c0).agg(F.count(F.lit(1)).alias("n")).toPandas()
                           .pivot(index=MONTH, columns=c0, values="n").fillna(0).astype(int).reset_index())
        with step("lineage against pkg_tm_cust_month"):
            _lin = pd.DataFrame()
            _mm = sorted(set(_kmap[MONTH].dropna()) & set(table_months(CONFIG["TM_TABLE"])))
            if _mm:
                mth = _mm[-1]
                cmp_ = [(a, b) for a, b in [("tm_rev_12_mnth", "tm_rev_12m"), ("gross_rev_12_mnth", "gross_rev_12m"),
                                            ("credit_rev_12_mnth", "credit_rev_12m"), ("non_credit_rev_12_mnth", "non_credit_rev_12m"),
                                            ("dhe", "dhe"), ("pd_percent_1_mnth", "pd_pct_now")]
                        if a in f12.columns and b in TMC]
                if cmp_:
                    jj = (f12.where(F.col(MONTH) == mth).select(ID, *[F.col(a).alias(f"f_{a}") for a, _ in cmp_])
                          .join(tm.where(F.col(MONTH) == mth).select(F.col(ID).cast("string").alias(ID),
                                                                     *[F.col(b).cast("double").alias(f"t_{b}") for _, b in cmp_]), ID))
                    r = jj.agg(F.count(F.lit(1)).alias("n"),
                               *[F.avg(within(f"t_{b}", F.col(f"f_{a}"), rel=0.01, abs_=1e-6).cast("double")).alias(f"eq_{i}")
                                 for i, (a, b) in enumerate(cmp_)],
                               *[F.expr(f"percentile_approx(t_{b} / f_{a}, 0.5, 1000)").alias(f"rt_{i}")
                                 for i, (a, b) in enumerate(cmp_)]).toPandas().iloc[0]
                    _lin = pd.DataFrame([dict(month=mth, fact_column=a, tm_column=b, customers=int(r.n),
                                              equal_share=r[f"eq_{i}"], median_ratio_tm_to_fact=r[f"rt_{i}"])
                                         for i, (a, b) in enumerate(cmp_)])
        f12.unpersist()
        fig, ax = plt.subplots(figsize=(11, 3.8))
        _kk = _kc.sort_values(K)
        ax.bar(_kk[K].astype(str), _kk["rows"], color=C1, width=0.8)
        ax.set_title(f"cdm_customer_fact_v: rows per {K}"); ax.set_ylabel("rows")
        ax.tick_params(axis="x", rotation=90, labelsize=7); ax.grid(axis="x", visible=False)
        savefig(fig, "f05_fact_rows_per_key", f"Rows per {K} in the customer fact view")
        CELL.show(("Discovery", _disc),
                  (f"Month keys: last 15 of {len(_kc)} {K} values", _kc.head(15)),
                  (f"Duplicates per customer x {K} (last {len(last)} keys)", _dpk),
                  ("Month key to calendar month (last keys)", _kmap),
                  ("Do 12 (3) monthly values add up to the 12 (3) month value?", _sums),
                  ("PD: share of consecutive month keys where the value changes", _pdcad),
                  ("PD date: age at the latest key (months)", _age),
                  ("Upgrade / downgrade column by month", _ud),
                  ("Lineage: pkg_tm_cust_month against the fact view, same customer-month", _lin))

## §8 · Verdict per column

Rules, applied in this order; a column can collect notes from the checks above as well.

| Verdict | Rule |
|---|---|
| empty | null in 99% of rows or more |
| constant | one distinct value |
| lagged copy | equals another column of an earlier month in 98% of rows or more (§3) |
| duplicate | rank correlation 0.995 or more with another column in levels and 0.98 or more in monthly changes (§4), or equals it within 5% (§5) |
| mostly zero | zero in 98% of non-null rows or more |
| slow | changes in fewer than 10% of consecutive customer-months: cannot time an event to the month |
| 12-month smoothed | trailing 12-month total: moves 1/12 of a monthly change; use a monthly column if one exists |
| use | none of the above |

`suggested_role` is a starting point for the contagion study and the PKG tables, to agree on, not a rule.

In [ ]:
CELL.start("§8 · Verdict per column")
ROLE = [(r"^pd_|^is_default$|past_due|iris_|watchlist", "credit outcome"),
        (r"^dhe$|^wrre$|^mre$|soft_exposure|credit_exposure_type", "credit exposure (weight)"),
        (r"_rev_|^ep_", "revenue outcome"),
        (r"analyzed_chrg|billed|waived|eca_used|ecr_amt", "TM fee outcome (monthly)"),
        (r"ftp_value|int_rate|interest_paid", "deposit value / pricing"),
        (r"ecr_rate", "pricing"),
        (r"bal|drawdown|overdraft|days_negative", "balance outcome"),
        (r"n_acct|acct_days", "account activity"),
        (r"^in_|^had_|^has_|^is_org$|cust_gone|n_fact_rows|n_days", "filter / data quality"),
        (rf"^{ID}$|^{MONTH}$", "key")]


def role(c):
    return next((r for p, r in ROLE if re.search(p, c)), "")


def verdicts(prof, chg, key):
    ch = chg.set_index("column")["change_share"] if len(chg) else pd.Series(dtype=float)
    rows = []
    for r in prof.itertuples():
        notes = VERDICT_NOTES.get((key, r.column), [])
        cs = ch.get(r.column, np.nan)
        if r.column in (ID, MONTH):
            v = "key"
        elif r.null_share >= 0.99:
            v = "empty"
        elif r.distinct <= 1:
            v = "constant"
        elif any(n.startswith("lagged copy") for n in notes):
            v = "lagged copy"
        elif any(n.startswith("near-duplicate") or n.startswith("matches deposits") or "redundant" in n for n in notes):
            v = "duplicate"
        elif pd.notna(getattr(r, "zero_share", np.nan)) and r.zero_share >= 0.98:
            v = "mostly zero"
        elif np.isfinite(cs) and cs < 0.10 and r.distinct > 2:
            v = "slow"
        elif re.search(r"_12m$|_12_mnth$", r.column):
            v = "12-month smoothed"
        else:
            v = "use"
        rows.append(dict(table=key, column=r.column, verdict=v, null_share=r.null_share,
                         change_share=cs, suggested_role=role(r.column), notes="; ".join(notes)))
    return pd.DataFrame(rows)


VER = [verdicts(PROF_TM, CHG_TM, "tm")]
if "dep" in SCHEMA:
    VER.append(verdicts(PROF_DEP, CHG_DEP, "dep"))
VER = pd.concat(VER, ignore_index=True)
VER.to_csv(os.path.join(OUT_DIR, "verdicts.csv"), index=False)
_sumv = VER.groupby(["table", "verdict"]).size().rename("columns").reset_index()
CELL.show(("Verdict counts", _sumv), ("Verdict per column (also in verdicts.csv)", VER),
          (f"Figures and CSVs in {OUT_DIR}", pd.DataFrame(FIG_INDEX, columns=["file", "caption"])),
          ("Timings (minutes)", pd.DataFrame(TIMINGS, columns=["step", "minutes"])))